# Experimento: Isolation Forest treinado em Fake

Esta variante compara duas refer?ncias de desvio de estilo com as mesmas seis features do notebook Isolation Forest original: um modelo ajustado somente em not?cias **True (0)** e outro ajustado somente em not?cias **Fake (1)**. Ambos usam os primeiros 300 caracteres, imputa??o pela mediana e Isolation Forest com 300 ?rvores.

**Hip?tese explorat?ria:** o desvio em rela??o ao estilo observado em uma classe pode ajudar a ordenar not?cias da outra classe, mas os dois grupos podem ter distribui??es e graus de dispers?o diferentes. Treinar em Fake n?o deve ser tratado como simples espelhamento do modelo treinado em True.

## Duas interpreta??es

- **IF ajustado em True:** score maior indica maior desvio em rela??o ?s not?cias True de treino. Acima do q95 de True-validation, a regra bin?ria sinaliza Fake.
- **IF ajustado em Fake:** score maior indica maior desvio em rela??o ?s not?cias Fake de treino. Acima do q95 de Fake-validation, sinaliza-se atipicidade em rela??o a Fake e, para a compara??o bin?ria, prev?-se True; abaixo do corte, prev?-se Fake.

Para ROC-AUC e Average Precision (AP), a classe positiva ? Fake: usamos o score original do IF treinado em True e **o score invertido** do IF treinado em Fake. Assim, score maior representa maior alinhamento com a ordena??o da classe Fake nessas m?tricas. Esses scores n?o s?o probabilidades de falsidade.

## Limites do experimento

As parti??es can?nicas por grupo e temporais v?m do manifesto indicado abaixo. Cada imputador e cada modelo s?o ajustados apenas com os registros da respectiva classe no treino; o q95 usa apenas a valida??o dessa mesma classe. Teste n?o participa do ajuste de features, modelo, limiar ou parti??o. Os protocolos s?o reportados separadamente.

As seis features s?o medidas lingu?sticas simples, n?o conte?do sem?ntico nem evid?ncia factual. Um score ou alerta n?o prova que uma not?cia seja verdadeira ou falsa. Os resultados s?o uma compara??o explorat?ria de desvio de estilo, sem teste de signific?ncia.

## 1. Imports e configura??o

A execu??o l? o Fake.br na mem?ria e n?o cria ZIP, CSV ou outro arquivo auxiliar. A ?nica sa?da persistente deste experimento ? este notebook com os outputs executados.

In [1]:
import hashlib
import io
import json
import platform
import re
import unicodedata
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
CHARACTER_LIMIT = 300
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn"]})

Python: 3.14.4
{'numpy': '2.5.3', 'pandas': '3.0.6', 'scikit-learn': '1.9.1'}


## 2. Carregamento do Fake.br

A revis?o do corpus ? a mesma do notebook original e do manifesto. O conte?do do ZIP ? mantido em mem?ria durante a execu??o. R?tulo e pasta de origem servem somente para auditoria, associa??o ?s parti??es e avalia??o; n?o entram na matriz de features.

In [2]:
MANIFEST_RELATIVE_PATH = Path(
    "machine-learning/outputs/model-comparison/"
    "dbscan-20260924T141332Z/run_manifest.json"
)
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / MANIFEST_RELATIVE_PATH).is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"N?o encontrei {MANIFEST_RELATIVE_PATH} a partir de {Path.cwd()}"
    )
MANIFEST_PATH = PROJECT_ROOT / MANIFEST_RELATIVE_PATH
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

corpus = manifest["corpus"]
corpus_url = corpus["url"]
with urlopen(corpus_url, timeout=180) as response:
    corpus_bytes = response.read()
corpus_sha256 = hashlib.sha256(corpus_bytes).hexdigest()
assert corpus_sha256 == corpus["expected_sha256"]

metadata_columns = [
    "autor", "link", "categoria", "data_publicacao", "num_tokens",
    "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos",
    "num_adjetivos", "num_adverbios", "num_verbos_modais",
    "num_pron_1_2_sing", "num_pron_1_plural", "num_pronomes",
    "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade",
    "diversidade",
]


def load_text_records(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        stem = Path(name).stem
        records.append({
            "id": stem + ("t" if label == 0 else ""),
            "text": archive.read(name).decode("utf-8"),
            "label": label,
            "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return records


def load_metadata_records(archive, folder, label):
    metadata_folder = f"/full_texts/{folder}-meta-information/"
    records = []
    for name in sorted(archive.namelist()):
        if metadata_folder not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadata_columns), (
            f"Esquema inesperado em {name}: {len(values)} linhas"
        )
        article_id = Path(name).name.removesuffix("-meta.txt")
        article_id += "t" if label == 0 else ""
        records.append({**dict(zip(metadata_columns, values)), "id": article_id,
                        "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return records


with ZipFile(io.BytesIO(corpus_bytes)) as archive:
    assert archive.testzip() is None, "ZIP do corpus corrompido"
    texts = (
        load_text_records(archive, "fake", 1)
        + load_text_records(archive, "true", 0)
    )
    metadata = (
        load_metadata_records(archive, "fake", 1)
        + load_metadata_records(archive, "true", 0)
    )

texts_frame = pd.DataFrame(texts)
metadata_frame = pd.DataFrame(metadata)
assert texts_frame["id"].is_unique and metadata_frame["id"].is_unique
assert set(texts_frame["id"]) == set(metadata_frame["id"])
news_frame = texts_frame.merge(
    metadata_frame, on="id", validate="one_to_one", indicator=True
)
assert news_frame["_merge"].eq("both").all()
assert news_frame["label"].eq(news_frame["metadataLabel"]).all()
news_frame = news_frame.drop(columns=["_merge", "metadataLabel"])
assert news_frame["text"].str.strip().ne("").all()
assert len(news_frame) == corpus["record_count"]
print(f"{len(news_frame):,} not?cias; SHA-256 verificado: {corpus_sha256}")

7,200 not?cias; SHA-256 verificado: be91c188f621424017bd79a0f33528dcc27f8a6151adb2bcb899c17719eb4090


## 3. Extra??o das seis features e integridade dos IDs

A extra??o reproduz o notebook original: normaliza??o Unicode NFKC, remo??o de BOM e truncamento antes da contagem. A matriz usa exatamente `tem_autor`, `typeTokenRatio`, `linkDensity`, `punctuationDensity`, `uppercaseRatio` e `diversidade`. IDs, grupo, r?tulo, nome de arquivo e `sourceClass` ficam fora das features.

Em fidelidade ? matriz original, `tem_autor` ? uma flag bin?ria de presen?a derivada do campo `autor`; o nome/identidade do autor n?o ? usado. ? a ?nica das seis features que n?o vem diretamente do prefixo de texto. Nenhum r?tulo, `sourceClass`, ID, nome de arquivo ou campo de metadado que carregue o r?tulo entra na matriz. A flag de presen?a pode refletir estilo editorial e deve ser interpretada como parte da limita??o explorat?ria.

In [3]:
FEATURES = [
    "tem_autor",
    "typeTokenRatio",
    "linkDensity",
    "punctuationDensity",
    "uppercaseRatio",
    "diversidade",
]
word_pattern = re.compile(r"[^\W\d_]+(?:['?\-][^\W\d_]+)*", re.UNICODE)
token_pattern = re.compile(
    r"[^\W\d_]+(?:['?\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]",
    re.UNICODE,
)


def safe_ratio(numerator, denominator):
    denominator = denominator if denominator > 0 else np.nan
    value = numerator / denominator
    return value if np.isfinite(value) else np.nan


def measure_prefix(text):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")[:CHARACTER_LIMIT]
    words = word_pattern.findall(normalized)
    tokens = token_pattern.findall(normalized)
    types = {word.casefold() for word in words}
    link_count = len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE))
    uppercase_count = sum(word.isupper() and len(word) > 1 for word in words)
    return {
        "num_palavras": len(words),
        "num_tokens": len(tokens),
        "num_types": len(types),
        "num_links": link_count,
        "num_maiusculas": uppercase_count,
        "num_caracteres": len(normalized),
    }


features_frame = news_frame[["id", "label", "sourceClass", "autor"]].copy()
features_frame["record_id"] = features_frame.apply(
    lambda row: f"{row['sourceClass']}/"
    f"{row['id'][:-1] if row['sourceClass'] == 'true' else row['id']}",
    axis=1,
)
author_values = features_frame["autor"].fillna("").astype(str).str.strip()
features_frame["tem_autor"] = (~author_values.isin(["", "None", "none", "NULL"])).astype(int)

measured = pd.DataFrame(
    [measure_prefix(text) for text in news_frame["text"]],
    index=features_frame.index,
)
for name in measured.columns:
    features_frame[name] = measured[name]
features_frame["typeTokenRatio"] = [
    safe_ratio(types, tokens)
    for types, tokens in zip(features_frame["num_types"], features_frame["num_tokens"])
]
features_frame["linkDensity"] = [
    safe_ratio(links, words)
    for links, words in zip(features_frame["num_links"], features_frame["num_palavras"])
]
features_frame["punctuationDensity"] = [
    safe_ratio(tokens - words, tokens)
    for tokens, words in zip(features_frame["num_tokens"], features_frame["num_palavras"])
]
features_frame["uppercaseRatio"] = [
    safe_ratio(uppercase, words)
    for uppercase, words in zip(features_frame["num_maiusculas"], features_frame["num_palavras"])
]
features_frame["diversidade"] = [
    safe_ratio(types, words)
    for types, words in zip(features_frame["num_types"], features_frame["num_palavras"])
]
features_frame[FEATURES] = features_frame[FEATURES].replace([np.inf, -np.inf], np.nan)

assert features_frame["record_id"].is_unique
assert set(features_frame["label"].unique()) == {0, 1}
assert features_frame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert "label" not in FEATURES and "id" not in FEATURES
assert "record_id" not in FEATURES and "sourceClass" not in FEATURES
assert "group_id" not in FEATURES and len(FEATURES) == 6
assert not np.isinf(features_frame[FEATURES].to_numpy(dtype=float)).any()
print("Features:", FEATURES)
display(features_frame.groupby("label").size().rename(index={0: "True (0)", 1: "Fake (1)"}).to_frame("not?cias"))

Features: ['tem_autor', 'typeTokenRatio', 'linkDensity', 'punctuationDensity', 'uppercaseRatio', 'diversidade']


,not?cias
label,
True (0),3600
Fake (1),3600


## 4. Parti??es can?nicas e temporais do manifesto

Os IDs e grupos s?o lidos da execu??o indicada: `record_ids` e parti??es v?m do `run_manifest.json`; a associa??o registro?grupo vem do `predictions.csv` desse mesmo diret?rio, que registra essa rela??o por protocolo e parti??o. Como o CSV repete registros por algoritmo, usamos apenas `protocol`, `partition`, `record_id` e `group_id`; nenhuma predi??o ou score pr?vio entra na an?lise. Cada rela??o ? conferida contra as listas de IDs e grupos do manifesto. N?o h? novo split.

O protocolo can?nico cobre os 7.200 registros. O holdout temporal usa os registros eleg?veis publicados no manifesto e mant?m fora um grupo (dois registros) sem data v?lida, conforme a auditoria temporal. As asser??es verificam a cobertura das listas publicadas, r?tulo indicado pelo prefixo de `record_id`, contagens e aus?ncia de grupo compartilhado entre parti??es do mesmo protocolo.


In [4]:
protocol_names = ["canonical_random_group", "secondary_temporal_group"]
assert all(name in manifest["partitions"] for name in protocol_names)
features_by_id = features_frame.set_index("record_id", drop=False)
group_audit_path = MANIFEST_PATH.parent / "predictions.csv"
group_audit = pd.read_csv(
    group_audit_path,
    usecols=["protocol", "partition", "record_id", "group_id"],
).drop_duplicates()
assert not group_audit.duplicated(["protocol", "partition", "record_id"]).any()
protocol_partitions = {}

for protocol_name in protocol_names:
    source_partitions = manifest["partitions"][protocol_name]
    current = {}
    all_protocol_ids = []
    group_sets = {}
    for split_name in ["train", "validation", "test"]:
        spec = source_partitions[split_name]
        record_ids = spec["record_ids"]
        expected_group_ids = spec["group_ids"]
        assert len(record_ids) == spec["counts"]["records"]
        assert len(record_ids) == len(set(record_ids))
        record_group_map = group_audit.loc[
            group_audit["protocol"].eq(protocol_name)
            & group_audit["partition"].eq(split_name),
            ["record_id", "group_id"],
        ].set_index("record_id")["group_id"]
        assert set(record_group_map.index) == set(record_ids)
        assert record_group_map.index.is_unique
        assert set(record_group_map.unique()) == set(expected_group_ids)
        assert record_group_map.nunique() == spec["counts"]["groups"]

        split_frame = features_by_id.loc[record_ids].copy()
        split_frame["group_id"] = record_group_map.loc[record_ids].to_numpy()
        assert split_frame["record_id"].tolist() == record_ids
        expected_label = split_frame["record_id"].str.startswith("true/").map({True: 0, False: 1})
        assert split_frame["label"].to_numpy().tolist() == expected_label.to_numpy().tolist()
        counts = split_frame["label"].value_counts().to_dict()
        assert counts.get(0, 0) == spec["counts"]["true_0"]
        assert counts.get(1, 0) == spec["counts"]["fake_1"]
        current[split_name] = split_frame.reset_index(drop=True)
        group_sets[split_name] = set(split_frame["group_id"])
        all_protocol_ids.extend(record_ids)
    assert len(all_protocol_ids) == len(set(all_protocol_ids))
    expected_protocol_records = sum(
        source_partitions[name]["counts"]["records"]
        for name in ["train", "validation", "test"]
    )
    assert len(all_protocol_ids) == expected_protocol_records
    assert set(all_protocol_ids).issubset(set(features_frame["record_id"]))
    if protocol_name == "canonical_random_group":
        assert set(all_protocol_ids) == set(features_frame["record_id"])
    else:
        temporal_audit = manifest["partition_audits"][protocol_name]
        assert temporal_audit["excluded_groups_missing_or_invalid_date"] == 1
        assert len(set(features_frame["record_id"]) - set(all_protocol_ids)) == 2
    for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        assert group_sets[left].isdisjoint(group_sets[right])
    protocol_partitions[protocol_name] = current

partition_summary = []
for protocol_name, parts in protocol_partitions.items():
    for split_name, frame in parts.items():
        partition_summary.append({
            "protocolo": protocol_name,
            "parti??o": split_name,
            "registros": len(frame),
            "grupos": frame["group_id"].nunique(),
            "True (0)": int(frame["label"].eq(0).sum()),
            "Fake (1)": int(frame["label"].eq(1).sum()),
        })
display(pd.DataFrame(partition_summary))


,protocolo,parti??o,registros,grupos,True (0),Fake (1)
0,canonical_random_group,train,4320,2159,2160,2160
1,canonical_random_group,validation,1440,720,720,720
2,canonical_random_group,test,1440,720,720,720
3,secondary_temporal_group,train,4304,2151,2152,2152
4,secondary_temporal_group,validation,1452,726,726,726
5,secondary_temporal_group,test,1442,721,721,721


## 5. Ajuste sim?trico e avalia??o

Cada protocolo ajusta dois pipelines independentes. O primeiro recebe somente True de treino e calibra o q95 em True-validation. O segundo recebe somente Fake de treino e calibra o q95 em Fake-validation. Ambos usam `SimpleImputer(strategy="median")` e `IsolationForest(n_estimators=300, contamination="auto", random_state=42, n_jobs=-1)`, sem `StandardScaler`.

A fronteira ? aplicada com `score >= q95`, como no notebook original; empates no percentil podem fazer mais de 5% da classe de valida??o ser sinalizada. Scores e limiares dos dois ajustes vivem em escalas pr?prias, ent?o os limiares n?o s?o comparados numericamente entre modelos.


In [5]:
def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("detector", IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


def anomaly_score(pipeline, frame):
    # Maior score = mais at?pico em rela??o ? classe de treino.
    return -pipeline.decision_function(frame[FEATURES])


def evaluate_protocol(protocol_name, parts):
    train = parts["train"]
    validation = parts["validation"]
    test = parts["test"]
    test_ids = test["record_id"].tolist()
    fits = {}

    for model_name, fit_label in [("IF treinado em True", 0), ("Variante treinada em Fake", 1)]:
        fit_frame = train.loc[train["label"].eq(fit_label)]
        assert not fit_frame.empty and fit_frame["label"].eq(fit_label).all()
        fit_features = fit_frame[FEATURES]
        assert not fit_features.columns[fit_features.isna().all()].any()
        fit_values = fit_features.to_numpy(dtype=float)
        assert np.isfinite(fit_values[~np.isnan(fit_values)]).all()

        pipeline = make_pipeline().fit(fit_features)
        assert list(pipeline.named_steps) == ["imputer", "detector"]
        assert pipeline.named_steps["detector"].n_estimators == 300
        np.testing.assert_allclose(
            pipeline.named_steps["imputer"].statistics_,
            fit_features.median().to_numpy(),
        )

        calibration_frame = validation.loc[validation["label"].eq(fit_label)]
        assert not calibration_frame.empty and calibration_frame["label"].eq(fit_label).all()
        calibration_scores = anomaly_score(pipeline, calibration_frame)
        threshold = float(np.quantile(calibration_scores, 0.95))
        raw_test_scores = anomaly_score(pipeline, test)
        fits[model_name] = {
            "fit_label": fit_label,
            "pipeline": pipeline,
            "threshold": threshold,
            "test_scores": raw_test_scores,
        }
        print(f"{protocol_name} | {model_name}: {len(fit_frame)} treino da classe {fit_label}; "
              f"{len(calibration_frame)} valida??o; q95={threshold:.6f}")

    original = fits["IF treinado em True"]
    variant = fits["Variante treinada em Fake"]
    assert len(original["test_scores"]) == len(variant["test_scores"]) == len(test)
    original_signal = original["test_scores"] >= original["threshold"]
    variant_signal = variant["test_scores"] >= variant["threshold"]
    predictions = {
        "IF treinado em True": original_signal.astype(int),
        # At?pico em rela??o a Fake => True; abaixo do corte => Fake.
        "Variante treinada em Fake": np.where(variant_signal, 0, 1),
    }
    signals = {
        "IF treinado em True": original_signal,
        "Variante treinada em Fake": variant_signal,
    }
    y_true = test["label"].to_numpy(dtype=int)
    summaries = {}
    for model_name, y_pred in predictions.items():
        raw_score = fits[model_name]["test_scores"]
        # Alinhar score alto ? classe positiva Fake em ROC-AUC/AP.
        fake_aligned_score = raw_score if model_name == "IF treinado em True" else -raw_score
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        summaries[model_name] = {
            "fit_label": fits[model_name]["fit_label"],
            "threshold": fits[model_name]["threshold"],
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "fake_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "fake_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            "fake_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            "fpr_true": fp / (tn + fp) if (tn + fp) else np.nan,
            "accuracy": accuracy_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, fake_aligned_score),
            "ap": average_precision_score(y_true, fake_aligned_score),
            "confusion": cm,
            "signal_counts": {
                "True": int(np.sum(signals[model_name] & (y_true == 0))),
                "Fake": int(np.sum(signals[model_name] & (y_true == 1))),
            },
        }
    return {
        "parts": parts,
        "test_ids": test_ids,
        "summaries": summaries,
        "predictions": predictions,
        "signals": signals,
    }


results = {
    name: evaluate_protocol(name, protocol_partitions[name])
    for name in protocol_names
}


canonical_random_group | IF treinado em True: 2160 treino da classe 0; 720 valida??o; q95=0.084177


canonical_random_group | Variante treinada em Fake: 2160 treino da classe 1; 720 valida??o; q95=0.080375


secondary_temporal_group | IF treinado em True: 2152 treino da classe 0; 726 valida??o; q95=0.103840


secondary_temporal_group | Variante treinada em Fake: 2152 treino da classe 1; 726 valida??o; q95=0.119274


## 6. Tabelas comparativas finais

As diferen?as s?o **variante menos IF treinado em True** e usam os mesmos itens de teste dentro de cada protocolo. A linha do limiar q95 fica sem diferen?a porque cada limiar est? na escala do seu pr?prio modelo. Nas matrizes, linhas s?o r?tulos reais e colunas s?o previstos, ambas na ordem **True, Fake**. Nas tabelas de sinaliza??o, ?sinalizado? significa acima do q95 da classe de treino correspondente.

In [6]:
def fmt_number(value, digits=4, signed=False):
    if isinstance(value, (int, np.integer)):
        return f"{value:+d}" if signed else str(value)
    if not np.isfinite(value):
        return "NaN"
    return f"{value:+.{digits}f}" if signed else f"{value:.{digits}f}"


def matrix_as_text(matrix, signed=False):
    values = matrix.ravel().tolist()
    if signed:
        parts = [f"{int(value):+d}" for value in values]
    else:
        parts = [str(int(value)) for value in values]
    tn, fp, fn, tp = parts
    return f"[[{tn}, {fp}], [{fn}, {tp}]]"


def build_comparison_table(protocol_result):
    summaries = protocol_result["summaries"]
    original = summaries["IF treinado em True"]
    variant = summaries["Variante treinada em Fake"]
    rows = [
        ("Classe usada no treino", "True", "Fake", "?", "classe usada no ajuste"),
        ("Limiar q95", fmt_number(original["threshold"], 6),
         fmt_number(variant["threshold"], 6), "?", "escalas pr?prias; n?o compar?vel"),
    ]
    metrics = [
        ("Macro-F1", "macro_f1"),
        ("Balanced accuracy", "balanced_accuracy"),
        ("Precis?o de Fake", "fake_precision"),
        ("Recall de Fake", "fake_recall"),
        ("F1 de Fake", "fake_f1"),
        ("FPR em True", "fpr_true"),
        ("Acur?cia", "accuracy"),
        ("ROC-AUC (score alinhado a Fake)", "roc_auc"),
        ("Average Precision (AP; score alinhado a Fake)", "ap"),
    ]
    for label, key in metrics:
        rows.append((label, fmt_number(original[key]), fmt_number(variant[key]),
                     fmt_number(variant[key] - original[key], signed=True), ""))
    matrix_delta = variant["confusion"] - original["confusion"]
    rows.append((
        "Matriz de confus?o ? linhas/colunas: True, Fake",
        matrix_as_text(original["confusion"]),
        matrix_as_text(variant["confusion"]),
        matrix_as_text(matrix_delta, signed=True),
        "[[TN, FP], [FN, TP]]; delta = variante ? original",
    ))
    return pd.DataFrame(rows, columns=[
        "M?trica", "IF treinado em True", "Variante treinada em Fake",
        "? (variante ? IF True)", "Nota",
    ])


def build_signal_table(protocol_result):
    rows = []
    summaries = protocol_result["summaries"]
    for model_name in ["IF treinado em True", "Variante treinada em Fake"]:
        counts = summaries[model_name]["signal_counts"]
        rows.append({
            "Modelo": model_name,
            "Sinal True": counts["True"],
            "Sinal Fake": counts["Fake"],
            "Sinal interpretado como": "Fake" if model_name == "IF treinado em True" else "True",
        })
    original, variant = rows
    rows.append({
        "Modelo": "? (variante ? IF True)",
        "Sinal True": variant["Sinal True"] - original["Sinal True"],
        "Sinal Fake": variant["Sinal Fake"] - original["Sinal Fake"],
        "Sinal interpretado como": "contagem acima do q95 pr?prio",
    })
    return pd.DataFrame(rows)


for protocol_name in protocol_names:
    display(Markdown(f"### Protocolo: `{protocol_name}` ? teste de {len(results[protocol_name]['test_ids']):,} registros"))
    display(build_comparison_table(results[protocol_name]))
    display(Markdown("**Itens sinalizados por classe real**"))
    display(build_signal_table(results[protocol_name]))

print(
    "Interpreta??o: comparar os valores ? explorat?rio. Um score ou alerta mede "
    "desvio em rela??o ? classe de treino e n?o verifica fatos."
)

### Protocolo: `canonical_random_group` ? teste de 1,440 registros

,M?trica,IF treinado em True,Variante treinada em Fake,? (variante ? IF True),Nota
0,Classe usada no treino,True,Fake,?,classe usada no ajuste
1,Limiar q95,0.084177,0.080375,?,escalas pr?prias; n?o compar?vel
2,Macro-F1,0.9729,0.9674,-0.0056,
3,Balanced accuracy,0.9729,0.9674,-0.0056,
4,Precis?o de Fake,0.9620,0.9828,+0.0208,
5,Recall de Fake,0.9847,0.9514,-0.0333,
6,F1 de Fake,0.9732,0.9668,-0.0064,
7,FPR em True,0.0389,0.0167,-0.0222,
8,Acur?cia,0.9729,0.9674,-0.0056,
9,ROC-AUC (score alinhado a Fake),0.9787,0.9716,-0.0071,


**Itens sinalizados por classe real**

,Modelo,Sinal True,Sinal Fake,Sinal interpretado como
0,IF treinado em True,28,709,Fake
1,Variante treinada em Fake,708,35,True
2,? (variante ? IF True),680,-674,contagem acima do q95 pr?prio


### Protocolo: `secondary_temporal_group` ? teste de 1,442 registros

,M?trica,IF treinado em True,Variante treinada em Fake,? (variante ? IF True),Nota
0,Classe usada no treino,True,Fake,?,classe usada no ajuste
1,Limiar q95,0.103840,0.119274,?,escalas pr?prias; n?o compar?vel
2,Macro-F1,0.8575,0.7383,-0.1192,
3,Balanced accuracy,0.8592,0.7524,-0.1068,
4,Precis?o de Fake,0.9609,0.6723,-0.2885,
5,Recall de Fake,0.7490,0.9847,+0.2358,
6,F1 de Fake,0.8418,0.7991,-0.0427,
7,FPR em True,0.0305,0.4799,+0.4494,
8,Acur?cia,0.8592,0.7524,-0.1068,
9,ROC-AUC (score alinhado a Fake),0.9829,0.9769,-0.0060,


**Itens sinalizados por classe real**

,Modelo,Sinal True,Sinal Fake,Sinal interpretado como
0,IF treinado em True,22,540,Fake
1,Variante treinada em Fake,375,11,True
2,? (variante ? IF True),353,-529,contagem acima do q95 pr?prio


Interpreta??o: comparar os valores ? explorat?rio. Um score ou alerta mede desvio em rela??o ? classe de treino e n?o verifica fatos.
